# 02 - Fine-tune RoBERTa (dùng GYAFC Official Test Set)

**Thay đổi so với phiên bản cũ:**
-  Train trên **100% dữ liệu train** (không chia 80/20 nữa)
-  Eval trên **tập test chính thức GYAFC** (`gyafc_test_ready.csv`)

| | Path |
|---|---|
| Train | `c:/Users/Admin/Documents/testvscode/formalityClassificationProject/data/gyafc_transformer_ready.csv` |
| Test  | `c:/Users/Admin/Documents/testvscode/formalityClassificationProject/data/gyafc_test_ready.csv` |
| Output| `c:/Users/Admin/Documents/testvscode/formalityClassificationProject/models/best_roberta_formality/` |

In [ ]:
!pip install transformers datasets evaluate accelerate scikit-learn


In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
import evaluate
import torch

# KIEM TRA GPU
if torch.cuda.is_available():
    print("GPU san sang!")
else:
    print("Dang dung CPU.")

# ────────────────────────────────────────────────────────
# 1. NAP DU LIEU
# ────────────────────────────────────────────────────────
print("\n1. DANG NAP DU LIEU...")
TRAIN_PATH = 'c:/Users/Admin/Documents/testvscode/formalityClassificationProject/data/gyafc_transformer_ready.csv'
TEST_PATH  = 'c:/Users/Admin/Documents/testvscode/formalityClassificationProject/data/gyafc_test_ready.csv'

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

# Dung 100% du lieu train (khong chia ngau nhien)
print(f"   Train : {len(train_df)} cau (100% training data)")
print(f"   Test  : {len(test_df)}  cau (GYAFC official test set)")

train_hg = Dataset.from_pandas(train_df[['text','label']].reset_index(drop=True))
test_hg  = Dataset.from_pandas(test_df[['text','label']].reset_index(drop=True))

# ────────────────────────────────────────────────────────
# 2. TOKENIZATION
# ────────────────────────────────────────────────────────
print("\n2. DANG MA HOA TU VUNG (TOKENIZATION)...")
model_name = "roberta-base"
tokenizer  = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length",
                     truncation=True, max_length=128)

train_tok = train_hg.map(tokenize_function, batched=True)
test_tok  = test_hg.map(tokenize_function, batched=True)

# ────────────────────────────────────────────────────────
# 3. MODEL & METRICS
# ────────────────────────────────────────────────────────
print("\n3. DANG KHOI TAO MODEL...")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

metric_acc = evaluate.load("accuracy")
metric_f1  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=preds, references=labels)
    f1  = metric_f1.compute(predictions=preds, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

# ────────────────────────────────────────────────────────
# 4. HUAN LUYEN (FINE-TUNING)
# ────────────────────────────────────────────────────────
print("\n4. BAT DAU HUAN LUYEN ROBERTA...")
training_args = TrainingArguments(
    output_dir="c:/Users/Admin/Documents/testvscode/formalityClassificationProject/models/roberta_formality_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),   # tu dong tat neu khong co GPU
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,    # TAP TEST CHINH THUC
    compute_metrics=compute_metrics,
)

trainer.train()

# ────────────────────────────────────────────────────────
# 5. LUU MODEL
# ────────────────────────────────────────────────────────
print("\n5. HUAN LUYEN HOAN TAT!")
trainer.save_model("c:/Users/Admin/Documents/testvscode/formalityClassificationProject/models/best_roberta_formality")
tokenizer.save_pretrained("c:/Users/Admin/Documents/testvscode/formalityClassificationProject/models/best_roberta_formality")
print("Model da luu tai: c:/Users/Admin/Documents/testvscode/formalityClassificationProject/models/best_roberta_formality")


In [ ]:
from transformers import pipeline
import torch

print("Dang khoi dong RoBERTa...")
classifier = pipeline(
    "text-classification",
    model="c:/Users/Admin/Documents/testvscode/formalityClassificationProject/models/best_roberta_formality",
    tokenizer="c:/Users/Admin/Documents/testvscode/formalityClassificationProject/models/best_roberta_formality",
    device=0 if torch.cuda.is_available() else -1
)

# --- THU NGHIEM ---
test_sentences = [
    "OMG!!! This is gonna be epic bro.",
    "I would like to request your assistance with this matter.",
    "u should totally check out that new movie lol",
    "It is imperative that we reconsider the current policy.",
    "I dunno what ur talking about tbh..."
]

print("\n--- KET QUA DU DOAN TU ROBERTA ---")
for text in test_sentences:
    result = classifier(text)[0]
    label_name = "Formal" if result['label'] == 'LABEL_1' else "Informal"
    print(f"  [{label_name}] {result['score']*100:.1f}%  ->  {text}")
